In [6]:
%reload_ext autoreload
%autoreload 2

import warnings, os, sys
warnings.filterwarnings('ignore')
from IPython.display import clear_output
from pathlib import Path

import numpy as np
import pandas as pd
import pickle
from pandarallel import pandarallel
pandarallel.initialize()
from tqdm.auto import tqdm
import json
import networkx as nx
from networkx.readwrite import json_graph

from rdkit import Chem
from rdkit.Chem import PandasTools
# PandasTools.RenderImagesInAllDataFrames(images=True)
from rdkit import RDLogger; RDLogger.DisableLog('rdApp.*')

from rdkit.Chem.Draw import IPythonConsole
IPythonConsole.ipython_useSVG = True
IPythonConsole.molSize = 700, 500

sys.path.append(os.path.abspath(".."))
from new_geass import Geass
import utils
clear_output()

In [12]:
df_NPA = pd.read_csv('/home/mattia/geass/data/NPAtlas_download_2024_09.tsv', sep='\t')
PandasTools.AddMoleculeColumnToFrame(df_NPA, 'compound_smiles')
df_NPA = df_NPA.dropna(subset='ROMol')

allowed_elements = {"H", "C", "N", "O", "F", "P", "S", "Cl", "Br", "I"}
df_NPA = df_NPA[df_NPA["ROMol"].apply(lambda mol: all(
    atom.GetSymbol() in allowed_elements for atom in mol.GetAtoms()))].reset_index(drop=True)

df_NPA['ROMol'] = df_NPA['ROMol'].parallel_map(utils.standardize_mol)
df_NPA["ISOSMILES"] = df_NPA["ROMol"].parallel_map(lambda x: Chem.MolToSmiles(x))
df_NPA = df_NPA.drop_duplicates(subset='ISOSMILES')
df_NPA = df_NPA.dropna(subset='ROMol').reset_index(drop=True)

num_cols = df_NPA.select_dtypes(include='number').columns
str_cols = df_NPA.columns.difference(num_cols)
df_NPA[str_cols] = df_NPA[str_cols].astype(str)
df_NPA.drop(columns=['ROMol']).to_csv('/home/mattia/geass/data/NPAtlas_filtered.csv', index=False)

In [ ]:
import threading
from concurrent.futures import ThreadPoolExecutor

_TLS = threading.local()

def _get_geass():
    geass = getattr(_TLS, 'geass', None)
    if geass is None:
        geass = Geass(fp='map4', sim_threshold=0.01, as_dict=True)
        _TLS.geass = geass
    return geass

def process_row(row):
    try:
        frags = _get_geass().get_fragments(row.ROMol)
        return {
            'NPA_ID': row.npaid,
            'ISOSMILES': row.compound_smiles,
            'Frag_ISOSMILES': [x['ISOSMILES'] for x in frags],
            'Frag_IDs': [x['MUID'] for x in frags],
            'Frag_Sim': [float(x['Similarity']) for x in frags],
            'Frag_Ann': [x['Annotation_status'] for x in frags],
            'Frag_Note': ','.join(x['Note'] for x in frags if x['Note'] is not None)
        }
    except Exception:
        return {
            'NPA_ID': row.npaid,
            'ISOSMILES': row.compound_smiles,
            'Frag_ISOSMILES': [],
            'Frag_IDs': [],
            'Frag_Sim': [],
            'Frag_Ann': [],
            'Frag_Note': None
        }

chunks = np.array_split(df_NPA, 50)
rows = []
with ThreadPoolExecutor(max_workers=8) as ex:
    for chunk in tqdm(chunks, total=len(chunks)):
        rows.extend(ex.map(process_row, [r for _, r in chunk.iterrows()]))

df_out = pd.DataFrame(rows)
df_out.to_parquet("NPA_fragments.parquet", index=False)
df_out

In [ ]:
# Reconstruct graph objects and annotate with unknown IDs
from networkx.readwrite import json_graph
graphs = {}
with open("NPA_graphs_map4.jsonl", "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split("\t", 1)
        if len(parts) != 2:
            continue
        npaid, g_json = parts
        data = json.loads(g_json)
        graphs[npaid] = json_graph.node_link_graph(data)

with open("NPA_graphs_map4.pkl", "wb") as f:
    pickle.dump(graphs, f, protocol=pickle.HIGHEST_PROTOCOL)